# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS42024/Shailesh-Flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [73]:
#My lane is a classification task because the model predicts whether a page is declining or not declining. The target variable, trend_direction, can be converted into two categories: declining (down) and not declining. The model learns from the available features and uses those patterns to classify each page. This helps identify which pages may need to be prioritized for a content refresh.


# Import pandas for working with the dataset.
import pandas as pd

# Import the function used to split data into
# training and testing sets.
from sklearn.model_selection import train_test_split

# Import the Random Forest machine-learning model.
from sklearn.ensemble import RandomForestClassifier

# Import metrics used to evaluate the model.
from sklearn.metrics import precision_score, classification_report



# 1. Load the dataset


# Read the FlyRank dataset.
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully.")
print("Total rows:", len(df))


#
# 2. Create the target variable (y)
# --------------------------------------------------

# "trend_direction" is what we want the model to predict.
#
# 1 = "down" → the page is declining
# 0 = anything else → the page is not declining
y = (df["trend_direction"] == "down").astype(int)

print("\nTarget distribution:")
print(y.value_counts())


# --------------------------------------------------
# 3. Create the feature variables (X)
# --------------------------------------------------

# Select only numerical columns.
# Random Forest needs numerical input.
numeric_columns = df.select_dtypes(
    include="number"
).columns.tolist()


# Remove content_id if it is a numerical column.
# It identifies the page and should not be used
# as a predictive feature.
if "content_id" in numeric_columns:
    numeric_columns.remove("content_id")


# Create X using the numerical features.
X = df[numeric_columns]

print("\nNumber of features:", X.shape[1])


# --------------------------------------------------
# 4. Split the data
# --------------------------------------------------

# Use 80% of the data for training
# and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


# --------------------------------------------------
# 5. Create the machine-learning model
# --------------------------------------------------

# Create a Random Forest classifier.
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


# --------------------------------------------------
# 6. Train the model
# --------------------------------------------------

# This is where the machine-learning model
# learns patterns from the training data.
model.fit(X_train, y_train)

print("Model training completed.")


# --------------------------------------------------
# 7. Make predictions
# --------------------------------------------------

# Use the trained model to predict whether
# test pages are declining.
predictions = model.predict(X_test)


# --------------------------------------------------
# 8. Evaluate the model
# --------------------------------------------------

# Calculate precision for the "down" class.
precision = precision_score(
    y_test,
    predictions,
    zero_division=0
)

print(f"\nPrecision: {precision:.3f}")


# Print a more detailed evaluation.
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)






Dataset loaded successfully.
Total rows: 30000

Target distribution:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Number of features: 30
Training rows: 24000
Testing rows: 6000
Model training completed.

Precision: 1.000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2748
           1       1.00      1.00      1.00      3252

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [74]:
#I'll predict is_declining_label (trend_direction == "down"), which is a defined rule calculated from the current window — not an observed future outcome — so it's a proxy label, not a true target.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


# Load the dataset so the machine-learning model can use the available page information.
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


# --------------------------------------------------
# Target / Proxy Label
# --------------------------------------------------

# The machine-learning model predicts whether a page is declining.
# The target comes from the "trend_direction" column in the dataset.
# "down" is treated as the positive label because it represents a declining page.
# This is a proxy label because it is based on a defined trend rule rather than a directly observed business outcome.
y = (df["trend_direction"] == "down").astype(int)


# --------------------------------------------------
# Features
# --------------------------------------------------

# The model uses the numerical features available in the dataset to identify patterns related to declining pages.
numeric_columns = df.select_dtypes(include="number").columns.tolist()

# Remove content_id because it identifies the page but does not provide useful information for prediction.
if "content_id" in numeric_columns:
    numeric_columns.remove("content_id")

# Store the selected features in X so the model can use them for prediction.
X = df[numeric_columns]


# --------------------------------------------------
# Training and Testing
# --------------------------------------------------

# Split the data so the model can learn from the training data and be evaluated on unseen data.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# --------------------------------------------------
# Machine Learning Model
# --------------------------------------------------

# Create a Random Forest model that can learn relationships between multiple features.
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Train the model so it can learn patterns associated with declining pages.
model.fit(X_train, y_train)

# Use the trained model to predict whether pages in the test data are declining.
predictions = model.predict(X_test)

# Display the model's predictions.
print("Machine-learning predictions:")
print(predictions[:20])

Machine-learning predictions:
[0 1 0 1 1 1 1 0 1 0 1 1 0 1 0 0 1 1 1 0]


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [61]:

#I'll Use The Precision@50 - of the top 50 pages my model ranks first, how many are actually labeled 'declining' - because the baseline rule scores 0.240 on this metric and I'd call anything meaningfully above that.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


# Load the FlyRank starter dataset.
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


# --------------------------------------------------
# Create the target variable (y)
# --------------------------------------------------

# Our target is "trend_direction".
# We treat "down" as the positive class.
# "down" becomes 1, while everything else becomes 0.
y = (df["trend_direction"] == "down").astype(int)


# --------------------------------------------------
# Create the feature variables (X)
# --------------------------------------------------

# These are the pieces of information the model
# will use to predict whether content is trending down.
X = df[
    [
        "content_age_days",
        "impressions_90d"
    ]
]


# --------------------------------------------------
# Split the data into training and testing sets
# --------------------------------------------------

# 80% of the data is used to train the model.
# 20% is kept separate to test the model.
# random_state=42 makes the split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# --------------------------------------------------
# Create the machine-learning model
# --------------------------------------------------

# Create a Random Forest classifier.
# The model will learn patterns from the training data
# that can help predict whether content is "down".
model = RandomForestClassifier(
    random_state=42
)


# --------------------------------------------------
# Train the model
# --------------------------------------------------

# Give the model the training features (X_train)
# and the correct answers (y_train).
# The model learns from these examples.
model.fit(X_train, y_train)


# --------------------------------------------------
# Make predictions
# --------------------------------------------------

# Use the trained model to predict the labels
# for the test data that the model has not seen before.
predictions = model.predict(X_test)


# Display a few predictions so we can check the result.
print("First 10 predictions:")
print(predictions[:10])

First 10 predictions:
[1 0 0 1 0 1 1 1 1 1]


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [65]:
#One row in content_refresh_anonymized.csv = one content page (content_id), with its search and engagement metrics rolled up over the same trailing 90-day window — confirmed below, since all 30,000 rows have a unique content_id.

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report



# 1. Load the dataset


# Read the FlyRank CSV file into a pandas DataFrame.
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")



# 2. Create the target variable (y)


# "trend_direction" is what we want the model to predict.

# We are treating "down" as the positive class:
# down = 1
# anything else = 0
y = (df["trend_direction"] == "down").astype(int)



# 3. Create the feature variables (X)
#
# These are the pieces of information the model
# will use to make its prediction.
X = df[
    [
        "content_age_days",
        "impressions_90d"
    ]
]


#
# 4. Split the data

# Split the dataset into training and testing data.
#
# 80% of the data will be used to train the model.
# 20% will be used to test the model.
#
# random_state=42 makes sure we get the same split
# every time we run the notebook.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)



# 5. Create the machine-learning model


# Create a Random Forest classification model.
#
# Random Forest uses multiple decision trees to learn
# patterns in the training data.
model = RandomForestClassifier(
    random_state=42
)



# 6. Train the model


# Give the model the training features and their
# corresponding labels.
#
# This is where the machine-learning model actually
# learns patterns from the data.
model.fit(X_train, y_train)



# 7. Make predictions


# Use the trained model to predict whether the
# test examples belong to the "down" class.
predictions = model.predict(X_test)



# 8. Evaluate the model


# Calculate the percentage of test predictions
# that were correct.
accuracy = accuracy_score(y_test, predictions)

print(f"Model accuracy: {accuracy:.3f}")


# Print additional performance information,
# including precision, recall, and F1-score.
print("\nClassification Report:")
print(classification_report(y_test, predictions))

Model accuracy: 0.612

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.58      0.58      2732
           1       0.65      0.63      0.64      3268

    accuracy                           0.61      6000
   macro avg       0.61      0.61      0.61      6000
weighted avg       0.61      0.61      0.61      6000



## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [68]:

#A fixed if-statement rule applies the same hand-picked weights to every page, but the signals interact differently case by case — one page has a great position (3.4) and CTR (0.89) yet the baseline still ranks it as urgent priority #19 even though it isn't declining, while another page with weak position (26.9) and CTR (0.09) is buried at baseline rank 8375 despite genuinely declining — and a fixed formula can't tell these two patterns apart the way a model trained on the interactions between all 52 features can.


import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, classification_report


# --------------------------------------------------
# 1. Load the dataset
# --------------------------------------------------

# Load the FlyRank starter dataset.
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


# --------------------------------------------------
# 2. Create the target variable (y)
# --------------------------------------------------

# "down" is our positive class.
# 1 = content is declining
# 0 = content is not declining
y = (df["trend_direction"] == "down").astype(int)


# --------------------------------------------------
# 3. Create the feature variables (X)
# --------------------------------------------------

# Select only numeric columns.
# Machine-learning models such as Random Forest
# require numerical input.
numeric_columns = df.select_dtypes(
    include="number"
).columns.tolist()


# Remove content_id if it is numeric.
# content_id identifies the page but should not be
# used as a predictive feature.
if "content_id" in numeric_columns:
    numeric_columns.remove("content_id")


# Create X using only the numeric feature columns.
X = df[numeric_columns]


# Print the number of features being used.
print("Number of numeric features:", X.shape[1])

print("\nFeatures being used:")
print(X.columns.tolist())


# --------------------------------------------------
# 4. Split the data
# --------------------------------------------------

# 80% of the data is used for training.
# 20% is used for testing.
#
# stratify=y keeps the proportion of declining and
# non-declining pages similar in both groups.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# --------------------------------------------------
# 5. Create the machine-learning model
# --------------------------------------------------

# Create a Random Forest classifier.
#
# The model can learn relationships between multiple
# features instead of using one fixed hand-written rule.
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


# --------------------------------------------------
# 6. Train the model
# --------------------------------------------------

# THIS IS THE MAIN MACHINE-LEARNING STEP.
#
# The model learns patterns from the training features
# and the known target labels.
model.fit(X_train, y_train)


# --------------------------------------------------
# 7. Make predictions
# --------------------------------------------------

# Use the trained model to predict which test pages
# are declining.
predictions = model.predict(X_test)


# --------------------------------------------------
# 8. Evaluate the model
# --------------------------------------------------

# Calculate precision.
# This tells us how many pages predicted as "down"
# were actually "down".
precision = precision_score(
    y_test,
    predictions,
    zero_division=0
)

print(f"\nPrecision: {precision:.3f}")


# Show additional model performance metrics.
print("\nClassification Report:")
print(classification_report(
    y_test,
    predictions,
    zero_division=0
))





Number of numeric features: 30

Features being used:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']

Precision: 1.000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2748
           1       1.00      1.00      1.00      3252

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.